# Part 2: Role-Based Access Control (RBAC) Module

This section implements comprehensive RBAC with role hierarchies, query normalization, access filtering, and validation.

In [13]:
print("🔐 Configuring Role Hierarchy and Permissions...\n")

# Define role hierarchy: higher index = more access
# C-Level at top > Department Leadership > Department Staff > General Employees
ROLE_HIERARCHY = {
    # C-Level (access to all departments)
    "admin": 100,
    "ceo": 100,
    "leadership": 90,
    
    # Department Leaders
    "engineering_lead": 80,
    "tech_lead": 80,
    "finance_lead": 80,
    "hr_lead": 80,
    "sales": 80,
    
    # Department Staff
    "engineering": 60,
    "finance": 60,
    "hr": 60,
    "marketing": 60,
    
    # General employees
    "all_employees": 40,
}

# Department access matrix: which roles can access which departments
DEPARTMENT_ACCESS = {
    "engineering": {
        "admin": True,
        "ceo": True,
        "leadership": True,
        "engineering_lead": True,
        "tech_lead": True,
        "engineering": True,
        "all_employees": False,  # General employees cannot see engineering
    },
    "finance": {
        "admin": True,
        "ceo": True,
        "leadership": True,
        "finance_lead": True,
        "finance": True,
        "all_employees": False,  # Restricted department
    },
    "hr": {
        "admin": True,
        "ceo": True,
        "leadership": True,
        "hr_lead": True,
        "hr": True,
        "all_employees": False,  # HR is restricted
    },
    "marketing": {
        "admin": True,
        "ceo": True,
        "leadership": True,
        "marketing": True,
        "sales": True,  # Sales can see marketing
        "all_employees": True,  # Marketing visible to all
    },
    "general": {
        "admin": True,
        "ceo": True,
        "leadership": True,
        "all_employees": True,  # Everyone can access general
    },
}

print("✅ Role Hierarchy Configured:")
for role, level in sorted(ROLE_HIERARCHY.items(), key=lambda x: -x[1])[:5]:
    print(f"   {role:20} -> Level {level}")
print(f"   ... and {len(ROLE_HIERARCHY)-5} more roles\n")

print("✅ Department Access Matrix:")
for dept, roles in DEPARTMENT_ACCESS.items():
    allowed_count = sum(1 for v in roles.values() if v)
    print(f"   {dept:20} -> {allowed_count} roles can access")


🔐 Configuring Role Hierarchy and Permissions...

✅ Role Hierarchy Configured:
   admin                -> Level 100
   ceo                  -> Level 100
   leadership           -> Level 90
   engineering_lead     -> Level 80
   tech_lead            -> Level 80
   ... and 8 more roles

✅ Department Access Matrix:
   engineering          -> 6 roles can access
   finance              -> 5 roles can access
   hr                   -> 5 roles can access
   marketing            -> 6 roles can access
   general              -> 4 roles can access


In [14]:
import re
import unicodedata
from typing import Set

print("📝 Query Preprocessing & Normalization Module\n")

def normalize_query(query: str) -> str:
    """
    Normalize query text:
    - Lowercase
    - Remove extra whitespace
    - Remove special characters (keep only alphanumeric and spaces)
    - Strip accents
    """
    # Lowercase
    query = query.lower()
    
    # Remove accents/diacritics
    query = ''.join(
        c for c in unicodedata.normalize('NFD', query)
        if unicodedata.category(c) != 'Mn'
    )
    
    # Remove special characters but keep words
    query = re.sub(r'[^a-z0-9\s]', ' ', query)
    
    # Collapse multiple spaces
    query = re.sub(r'\s+', ' ', query).strip()
    
    return query


def extract_keywords(query: str, min_length: int = 3) -> Set[str]:
    """
    Extract meaningful keywords from query.
    """
    normalized = normalize_query(query)
    keywords = set(word for word in normalized.split() if len(word) >= min_length)
    return keywords


def expand_query_with_synonyms(query: str) -> str:
    """
    Expand query with common synonyms for better matching.
    """
    synonyms = {
        "revenue": "sales income earnings",
        "profit": "earnings gains returns",
        "financial": "monetary fiscal accounting",
        "engineering": "technical development software",
        "hr": "human resources staff personnel",
        "marketing": "promotion advertising campaign",
        "strategy": "plan approach tactics",
        "performance": "results metrics effectiveness",
        "policy": "rules guidelines procedures",
        "best practice": "standard methodology guideline",
    }
    
    expanded = query.lower()
    for key, value in synonyms.items():
        if key in expanded:
            expanded += f" {value}"
    
    return expanded


print("✅ Query utilities defined:")
print("   - normalize_query(): Standardize query text")
print("   - extract_keywords(): Extract meaningful keywords")
print("   - expand_query_with_synonyms(): Expand with synonyms")

# Test utilities
test_query = "What are the Q4 Financial Results?"
print(f"\n📊 Test query: '{test_query}'")
print(f"   Normalized: '{normalize_query(test_query)}'")
print(f"   Keywords: {extract_keywords(test_query)}")
print(f"   Expanded: '{expand_query_with_synonyms(test_query)[:100]}...'")


📝 Query Preprocessing & Normalization Module

✅ Query utilities defined:
   - normalize_query(): Standardize query text
   - extract_keywords(): Extract meaningful keywords
   - expand_query_with_synonyms(): Expand with synonyms

📊 Test query: 'What are the Q4 Financial Results?'
   Normalized: 'what are the q4 financial results'
   Keywords: {'financial', 'are', 'what', 'results', 'the'}
   Expanded: 'what are the q4 financial results? monetary fiscal accounting...'


In [15]:
from pathlib import Path
from typing import List, Dict
import pandas as pd

print("🔐 RBAC Filtering Logic Module\n")


def _resolve_chunks_csv() -> Path:
    """Resolve the chunk artifact path from notebook or workspace root."""
    candidates = [
        Path.cwd() / "artifacts" / "module2" / "chunks" / "cleaned_formatted_chunks.csv",
        Path.cwd().parent / "artifacts" / "module2" / "chunks" / "cleaned_formatted_chunks.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not locate cleaned_formatted_chunks.csv in expected artifact paths.")


chunks_csv_path = _resolve_chunks_csv()
chunks_df = pd.read_csv(chunks_csv_path)
chunks: List[Dict] = chunks_df.to_dict("records")

# Normalize role field from CSV string to list
for chunk in chunks:
    raw_roles = chunk.get("accessible_roles", "")
    if isinstance(raw_roles, str):
        chunk["accessible_roles"] = [r.strip() for r in raw_roles.split(",") if r.strip()]

print(f"✅ Loaded chunks for RBAC filtering: {len(chunks)}")
print(f"   Source: {chunks_csv_path}")


class RBACAccessControl:
    """Role-Based Access Control manager for document chunks."""

    def __init__(self, role_hierarchy: Dict[str, int], dept_access: Dict[str, Dict[str, bool]]):
        self.role_hierarchy = role_hierarchy
        self.dept_access = dept_access

    def get_user_access_level(self, user_roles: List[str]) -> int:
        """Get highest access level for user based on their roles."""
        return max((self.role_hierarchy.get(role, 0) for role in user_roles), default=0)

    def is_privileged(self, user_roles: List[str]) -> bool:
        """Privileged roles bypass chunk-level role tags once department policy allows access."""
        return self.get_user_access_level(user_roles) >= 90

    def can_user_access_department(self, user_roles: List[str], department: str) -> bool:
        """Check if user can access a specific department."""
        dept_perms = self.dept_access.get(str(department).lower(), {})
        return any(dept_perms.get(role, False) for role in user_roles)

    def can_user_access_chunk(self, user_roles: List[str], chunk: Dict) -> bool:
        """Check if user can access a specific chunk using department + chunk role checks."""
        chunk_dept = str(chunk.get("department", "")).lower()
        if not self.can_user_access_department(user_roles, chunk_dept):
            return False

        # Hierarchy override: C-level and equivalent roles can read all chunks in authorized departments.
        if self.is_privileged(user_roles):
            return True

        chunk_roles = chunk.get("accessible_roles", [])
        if chunk_roles:
            return any(role in chunk_roles for role in user_roles)

        return True

    def filter_chunks_by_role(self, user_roles: List[str], chunk_records: List[Dict]) -> List[Dict]:
        """Filter list of chunks by user access."""
        return [chunk for chunk in chunk_records if self.can_user_access_chunk(user_roles, chunk)]

    def get_accessible_departments(self, user_roles: List[str]) -> List[str]:
        """Get list of departments user can access."""
        accessible = [
            dept for dept in self.dept_access.keys() if self.can_user_access_department(user_roles, dept)
        ]
        return sorted(accessible)

    def get_role_summary(self, user_roles: List[str], chunk_records: List[Dict]) -> Dict:
        """Get summary of a user's access rights and visibility."""
        visible = self.filter_chunks_by_role(user_roles, chunk_records)
        return {
            "roles": user_roles,
            "access_level": self.get_user_access_level(user_roles),
            "accessible_departments": self.get_accessible_departments(user_roles),
            "total_chunks_visible": len(visible),
        }


# Initialize RBAC system
rbac = RBACAccessControl(ROLE_HIERARCHY, DEPARTMENT_ACCESS)

print("\n✅ RBAC Access Control System Initialized")
print("\nTesting access for different roles:")

test_users = {
    "C-Level": ["ceo"],
    "Finance Manager": ["finance_lead", "leadership"],
    "Engineer": ["engineering"],
    "HR Staff": ["hr"],
    "General Employee": ["all_employees"],
}

for user_title, user_roles in test_users.items():
    summary = rbac.get_role_summary(user_roles, chunks)
    total = len(chunks)
    accessible = summary["total_chunks_visible"]
    pct = int((100 * accessible / total)) if total else 0
    print(f"\n{user_title:20} (roles: {', '.join(user_roles)})")
    print(f"   Access Level: {summary['access_level']}")
    print(f"   Can access: {', '.join(summary['accessible_departments'])}")
    print(f"   Visible chunks: {accessible}/{total} ({pct}%)")

🔐 RBAC Filtering Logic Module

✅ Loaded chunks for RBAC filtering: 311
   Source: d:\Projects\RBAC\artifacts\module2\chunks\cleaned_formatted_chunks.csv

✅ RBAC Access Control System Initialized

Testing access for different roles:

C-Level              (roles: ceo)
   Access Level: 100
   Can access: engineering, finance, general, hr, marketing
   Visible chunks: 311/311 (100%)

Finance Manager      (roles: finance_lead, leadership)
   Access Level: 90
   Can access: engineering, finance, general, hr, marketing
   Visible chunks: 311/311 (100%)

Engineer             (roles: engineering)
   Access Level: 60
   Can access: engineering
   Visible chunks: 77/311 (24%)

HR Staff             (roles: hr)
   Access Level: 60
   Can access: hr
   Visible chunks: 48/311 (15%)

General Employee     (roles: all_employees)
   Access Level: 40
   Can access: general, marketing
   Visible chunks: 38/311 (12%)


In [16]:
print("🔎 Query Processing, Relevance Selection, and RBAC Validation\n")


def score_chunk_relevance(query: str, chunk: Dict) -> float:
    """Simple semantic-lite scoring based on normalized text overlap and phrase presence."""
    normalized_query = normalize_query(expand_query_with_synonyms(query))
    query_terms = [t for t in normalized_query.split() if len(t) >= 3]
    if not query_terms:
        return 0.0

    content = normalize_query(str(chunk.get("content", "")))
    if not content:
        return 0.0

    content_terms = set(content.split())
    overlap = sum(1 for term in query_terms if term in content_terms)
    overlap_ratio = overlap / max(len(set(query_terms)), 1)

    phrase_bonus = 0.15 if normalize_query(query) in content else 0.0

    dept = str(chunk.get("department", "")).lower()
    department_hint_bonus = 0.0
    if "finance" in normalized_query and dept == "finance":
        department_hint_bonus = 0.1
    elif "hr" in normalized_query and dept == "hr":
        department_hint_bonus = 0.1
    elif "engineering" in normalized_query and dept == "engineering":
        department_hint_bonus = 0.1
    elif "marketing" in normalized_query and dept == "marketing":
        department_hint_bonus = 0.1

    return round(min(1.0, overlap_ratio + phrase_bonus + department_hint_bonus), 4)


def select_relevant_chunks(
    query: str,
    user_roles: List[str],
    chunk_records: List[Dict],
    top_k: int = 5,
) -> List[Dict]:
    """RBAC-aware retrieval: filter first, then rank for relevance."""
    filtered = rbac.filter_chunks_by_role(user_roles, chunk_records)

    ranked = []
    for chunk in filtered:
        score = score_chunk_relevance(query, chunk)
        if score > 0:
            enriched = dict(chunk)
            enriched["relevance_score"] = score
            ranked.append(enriched)

    ranked.sort(key=lambda item: item["relevance_score"], reverse=True)
    return ranked[:top_k]


# Query processing demonstration
demo_queries = [
    {"query": "Q4 financial results and revenue", "roles": ["finance"]},
    {"query": "employee leave policy and benefits", "roles": ["hr"]},
    {"query": "engineering architecture best practices", "roles": ["engineering"]},
    {"query": "marketing campaign strategy", "roles": ["all_employees"]},
]

print("✅ Running RBAC-aware relevance selection demo:\n")
for i, item in enumerate(demo_queries, start=1):
    query = item["query"]
    roles = item["roles"]
    results = select_relevant_chunks(query=query, user_roles=roles, chunk_records=chunks, top_k=3)

    print(f"{i}. Query: {query}")
    print(f"   Roles: {roles}")
    print(f"   Returned chunks: {len(results)}")
    for r in results:
        preview = str(r.get("content", ""))[:90].replace("\n", " ")
        print(
            f"   - {r.get('chunk_id')} | dept={r.get('department')} | score={r.get('relevance_score')} | {preview}..."
        )
    print()


# Validation tests for role-based isolation
print("🧪 RBAC Validation Tests:\n")

validation_cases = [
    {
        "name": "Finance user cannot access HR",
        "roles": ["finance"],
        "forbidden_department": "hr",
    },
    {
        "name": "HR user cannot access Finance",
        "roles": ["hr"],
        "forbidden_department": "finance",
    },
    {
        "name": "Engineering user cannot access HR",
        "roles": ["engineering"],
        "forbidden_department": "hr",
    },
    {
        "name": "General employee cannot access Finance",
        "roles": ["all_employees"],
        "forbidden_department": "finance",
    },
    {
        "name": "C-Level can access all core departments",
        "roles": ["ceo"],
        "required_departments": ["engineering", "finance", "hr", "marketing", "general"],
    },
]

validation_results = []
for case in validation_cases:
    name = case["name"]
    roles = case["roles"]
    visible = rbac.filter_chunks_by_role(roles, chunks)
    visible_departments = {str(c.get("department", "")).lower() for c in visible}

    if "forbidden_department" in case:
        forbidden = case["forbidden_department"]
        passed = forbidden not in visible_departments
        details = f"forbidden={forbidden}, visible_departments={sorted(visible_departments)}"
    else:
        required = set(case["required_departments"])
        passed = required.issubset(visible_departments)
        missing = sorted(required - visible_departments)
        details = f"required={sorted(required)}, missing={missing}"

    validation_results.append({
        "test": name,
        "roles": roles,
        "passed": passed,
        "details": details,
    })

for row in validation_results:
    status = "PASS" if row["passed"] else "FAIL"
    print(f"[{status}] {row['test']} | roles={row['roles']}")
    print(f"       {row['details']}")

passed_count = sum(1 for row in validation_results if row["passed"])
print(f"\nValidation summary: {passed_count}/{len(validation_results)} tests passed")

# Persist test report for deliverable
results_path = Path.cwd() / "artifacts" / "module2" / "rbac_validation_results.csv"
if not results_path.parent.exists():
    results_path = Path.cwd().parent / "artifacts" / "module2" / "rbac_validation_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(validation_results).to_csv(results_path, index=False)
print(f"✅ Validation results saved: {results_path}")

🔎 Query Processing, Relevance Selection, and RBAC Validation

✅ Running RBAC-aware relevance selection demo:

1. Query: Q4 financial results and revenue
   Roles: ['finance']
   Returned chunks: 3
   - CHUNK-000100 | dept=finance | score=0.4 | increase from 2.1 billion in q1 to 2.6 billion in q4, alongside consistent improvements in...
   - CHUNK-000120 | dept=finance | score=0.4 | flecting optimized pricing and operational efficiencies. - operating income: 650 million, ...
   - CHUNK-000124 | dept=finance | score=0.4 | procurement processes and negotiated fixed-cost agreements to control expenses. - risk: re...

2. Query: employee leave policy and benefits
   Roles: ['hr']
   Returned chunks: 0

3. Query: engineering architecture best practices
   Roles: ['engineering']
   Returned chunks: 3
   - CHUNK-000002 | dept=engineering | score=0.5 | th management, and enterprise financial analytics, serving over 2 million individual users...
   - CHUNK-000003 | dept=engineering | score=0.5 | 